## 1️⃣ Imports

In [ ]:
import os
import io
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

print("TensorFlow version:", tf.__version__)


## 2️⃣ (Optional) Mount Google Drive

If you're running this in **Google Colab** and your model / test images live on
Drive (as in the original training notebooks), mount it here. If you're running
locally, this cell is safely skipped.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not running in Google Colab — skipping Drive mount.")


## 3️⃣ Load the Trained Model

Same model artifact used in `grad_cam_skin_disease.ipynb`.

In [ ]:
# Path to the full trained model (architecture + weights), saved at the end of training
MODEL_PATH = "/content/drive/MyDrive/defense/save/ResNet50V2_Blur_FullModel.keras"

# Fallback: allow overriding the path if the file lives elsewhere (e.g. local run)
if not IN_COLAB and not os.path.exists(MODEL_PATH):
    MODEL_PATH = input("Model not found at the default Drive path. "
                        "Enter the full path to 'ResNet50V2_Blur_FullModel.keras': ").strip()

print("Loading model from:", MODEL_PATH)
model = load_model(MODEL_PATH)
print("Model loaded successfully ✅")


## 4️⃣ Class Labels

Must match the alphabetical order Keras assigned during training (`flow_from_directory` / `test_datagen.class_indices`).

In [ ]:
class_labels = [
    'Acne',
    'Dermatophyte Infections Ringworms',
    'Eczema',
    'Hair Diseases',
    'Insecets Bite and Sting',
    'Nail Disease',
    'Pigment Disorder',
    'Skin Cancer',
    'Urticaria Skin Alergey',
    'Vascular Tumors',
    'Vasculitis',
    'Versicolor'
]
print(f"{len(class_labels)} classes loaded.")


## 5️⃣ User Selects an Image



In [ ]:
def get_image_path():
    """Lets the user pick an image either by typing a path or by uploading a file."""
    print("How would you like to provide the image?")
    print("  1) Enter an image path (development/testing)")
    print("  2) Upload an image from your computer (presentation)")
    choice = input("Enter 1 or 2: ").strip()

    if choice == "1":
        img_path = input("Enter the full path to the image: ").strip()
        while not os.path.exists(img_path):
            img_path = input("File not found. Please re-enter a valid path: ").strip()
        return img_path

    elif choice == "2":
        if IN_COLAB:
            from google.colab import files
            uploaded = files.upload()
            img_path = list(uploaded.keys())[0]
            return img_path
        else:
            print("File upload widget requires Google Colab.")
            img_path = input("Falling back to manual path entry — enter image path: ").strip()
            return img_path

    else:
        print("Invalid choice, defaulting to path entry.")
        return get_image_path()

# Uncomment to run interactively:
# selected_image_path = get_image_path()


## 6️⃣ Preprocess Image

Resize to `224×224` and normalize pixel values to `[0, 1]`, exactly as during training/evaluation.

In [ ]:
IMG_SIZE = (224, 224)

def preprocess_image(img_path, target_size=IMG_SIZE):
    """Loads an image and returns (PIL image for display, normalized model input array)."""
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    return img, img_array


In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import (
    MobileNetV2, preprocess_input as imagenet_preprocess, decode_predictions
)

print("Loading lightweight ImageNet sanity-check model "
      "(first run may take a moment to download pretrained weights)...")
ood_guard_model = MobileNetV2(weights="imagenet", include_top=True)
print("OOD guard model ready ✅")


In [ ]:
# Keyword fragments that signal "this is clearly not a skin close-up photo".
# Not exhaustive by design -- it only needs to catch obviously wrong uploads
# (animals, vehicles, scenery, everyday objects), not borderline cases.
NON_SKIN_KEYWORDS = [
    "cat", "kitten", "tabby", "persian", "dog", "puppy", "labrador", "poodle",
    "bird", "fish", "shark", "snake", "lizard", "frog", "insect", "butterfly", "spider",
    "car", "truck", "bus", "bicycle", "motorcycle", "train", "airplane", "boat", "ship",
    "mountain", "valley", "seashore", "lakeside", "volcano", "cliff", "forest", "sky", "alp",
    "building", "castle", "church", "bridge", "skyscraper", "house", "barn",
    "chair", "table", "sofa", "bed", "furniture",
    "pizza", "burger", "fruit", "vegetable", "plate", "food",
    "flower", "tree", "plant", "leaf",
    "guitar", "piano", "violin", "instrument",
    "keyboard", "laptop", "phone", "television", "monitor",
    "ball", "racket", "bat", "helmet",
]

OOD_CONFIDENCE_THRESHOLD = 0.30      # how sure the ImageNet gate must be to reject
SANITY_CONFIDENCE_THRESHOLD = 0.30   # below this, treat our own model's call as unreliable


def check_is_skin_image(img_pil):
    """
    Stage-1 OOD gate. Runs a general-purpose ImageNet classifier on the raw
    image and rejects it if the top prediction confidently matches a
    non-dermatological object/scene category.
    Returns (is_skin_image: bool, reason: str, imagenet_top3: list).
    """
    resized = img_pil.resize((224, 224))
    arr = image.img_to_array(resized)
    arr = np.expand_dims(arr, axis=0)
    arr = imagenet_preprocess(arr)

    preds = ood_guard_model.predict(arr, verbose=0)
    top3 = decode_predictions(preds, top=3)[0]   # [(class_id, label, prob), ...]
    top1_label, top1_prob = top3[0][1], float(top3[0][2])

    label_lower = top1_label.lower().replace("_", " ")
    looks_non_skin = any(kw in label_lower for kw in NON_SKIN_KEYWORDS)

    if looks_non_skin and top1_prob >= OOD_CONFIDENCE_THRESHOLD:
        reason = (f"This looks like '{top1_label.replace('_', ' ')}' "
                   f"({top1_prob:.1%} confidence) rather than a skin close-up.")
        return False, reason, top3

    return True, "Passed sanity check.", top3


def display_ood_result(is_skin_image, reason, top3):
    if is_skin_image:
        print("🟢 Sanity check passed — proceeding to diagnosis.")
    else:
        print("🔴 Sanity check failed — this does not look like a skin image.")
        print(f"   Reason: {reason}")
        print("   ImageNet's best guesses for this photo:")
        for _, label, prob in top3:
            print(f"     - {label.replace('_', ' '):<25s} {prob:.1%}")
        print("\n⚠️  Please upload a clear, close-up photo of the affected skin area.")


## 7️⃣ Model Prediction

In [ ]:
def predict_disease(img_array):
    """Runs inference and returns predicted class, confidence, and full probability vector."""
    preds = model.predict(img_array, verbose=0)[0]
    pred_idx = int(np.argmax(preds))
    pred_label = class_labels[pred_idx]
    confidence = float(preds[pred_idx])
    return pred_label, confidence, preds


## 8️⃣ Display Prediction, Confidence & Decision

If confidence is **≥ 70%** the prediction is accepted; otherwise the pipeline recommends consulting a dermatologist.

In [ ]:
CONFIDENCE_THRESHOLD = 0.70

def display_prediction(pred_label, confidence):
    print("=" * 50)
    print(f"🔍 Predicted Disease : {pred_label}")
    print(f"📊 Confidence Score  : {confidence:.2%}")
    print("=" * 50)

    if confidence >= CONFIDENCE_THRESHOLD:
        decision = "Prediction Accepted ✅"
        print(f"Decision: {decision}  (confidence ≥ {CONFIDENCE_THRESHOLD:.0%})")
    else:
        decision = "Consult a Dermatologist ⚠️"
        print(f"Decision: {decision}  (confidence < {CONFIDENCE_THRESHOLD:.0%})")
        print("The model isn't confident enough — please seek a professional opinion.")
    return decision


## 9️⃣ Top‑3 Predictions

In [ ]:
def get_top3(preds):
    top3_idx = np.argsort(preds)[::-1][:3]
    return [(class_labels[i], float(preds[i])) for i in top3_idx]

def display_top3(top3):
    print("Top‑3 Predictions:")
    for rank, (label, conf) in enumerate(top3, start=1):
        print(f"  {rank}. {label:<35s} {conf:.2%}")

    # Bar chart visualization
    labels = [t[0] for t in top3]
    scores = [t[1] * 100 for t in top3]
    colors = ["#2ecc71", "#3498db", "#95a5a6"]

    plt.figure(figsize=(6, 3.5))
    bars = plt.barh(labels[::-1], scores[::-1], color=colors[::-1])
    plt.xlabel("Confidence (%)")
    plt.title("Top‑3 Predictions")
    plt.xlim(0, 100)
    for bar, score in zip(bars, scores[::-1]):
        plt.text(score + 1, bar.get_y() + bar.get_height() / 2, f"{score:.1f}%",
                  va="center", fontsize=9)
    plt.tight_layout()
    plt.show()


## 🔟 Grad‑CAM Explainability

Reused directly from `grad_cam_skin_disease.ipynb`.

In [ ]:
# Locate the nested ResNet50V2 sub-model inside the classifier
base_model_name = 'resnet50v2'
base_model = model.get_layer(base_model_name)

# Last conv/activation layer of ResNet50V2 before global average pooling
last_conv_layer_name = 'post_relu'

# Everything in the classifier AFTER the base model, in order
# (e.g. [global_average_pooling2d, dense, dense_1, dense_2, dense_3])
head_layers = model.layers[2:]

def generate_grad_cam(img_array, class_idx):
    img_tensor = tf.convert_to_tensor(img_array)

    with tf.GradientTape() as tape:
        conv_outputs = base_model(img_tensor)      # (1, 7, 7, 2048)
        tape.watch(conv_outputs)

        x = conv_outputs
        for layer in head_layers:
            x = layer(x)
        predictions = x                            # (1, num_classes)

        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)[0]

    # Importance weights via global average pooling of the gradients
    weights = tf.reduce_mean(grads, axis=(0, 1))

    # Weighted sum of the activation maps
    cam = tf.reduce_sum(tf.multiply(weights, conv_outputs[0]), axis=-1)

    # ReLU to keep only features that positively influence the predicted class
    cam = tf.nn.relu(cam)

    # Resize heatmap to the input image size
    cam = tf.image.resize(cam[tf.newaxis, ..., tf.newaxis], IMG_SIZE)

    # Min-max normalize to [0, 1]
    cam_min, cam_max = tf.reduce_min(cam), tf.reduce_max(cam)
    cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)

    return cam.numpy()


## 1️⃣1️⃣ Display Original Image, Heatmap & Overlay

In [ ]:
def display_grad_cam(img, cam, pred_label, confidence):
    plt.figure(figsize=(14, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title(f"Original Image\nPredicted: {pred_label} ({confidence:.2%})")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(cam[0, ..., 0], cmap='jet', vmin=0, vmax=1)
    plt.title('Grad-CAM Heatmap')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(img)
    plt.imshow(cam[0, ..., 0], cmap='jet', alpha=0.5, vmin=0, vmax=1)
    plt.title('Grad-CAM Overlay')
    plt.axis('off')

    plt.tight_layout()
    plt.show()


## One‑Click Diagnostic Report



In [ ]:
def save_diagnostic_report(img, cam, top3, pred_label, confidence, decision,
                            out_path=None):
    if out_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = f"diagnostic_report_{timestamp}.png"

    fig = plt.figure(figsize=(14, 8))
    fig.suptitle("Skin Disease Diagnostic Report", fontsize=16, fontweight='bold')

    # Original image
    ax1 = plt.subplot(2, 3, 1)
    ax1.imshow(img)
    ax1.set_title("Original Image")
    ax1.axis('off')

    # Heatmap
    ax2 = plt.subplot(2, 3, 2)
    ax2.imshow(cam[0, ..., 0], cmap='jet', vmin=0, vmax=1)
    ax2.set_title("Grad-CAM Heatmap")
    ax2.axis('off')

    # Overlay
    ax3 = plt.subplot(2, 3, 3)
    ax3.imshow(img)
    ax3.imshow(cam[0, ..., 0], cmap='jet', alpha=0.5, vmin=0, vmax=1)
    ax3.set_title("Grad-CAM Overlay")
    ax3.axis('off')

    # Top-3 bar chart
    ax4 = plt.subplot(2, 1, 2)
    labels = [t[0] for t in top3]
    scores = [t[1] * 100 for t in top3]
    colors = ["#2ecc71", "#3498db", "#95a5a6"]
    bars = ax4.barh(labels[::-1], scores[::-1], color=colors[::-1])
    ax4.set_xlim(0, 100)
    ax4.set_xlabel("Confidence (%)")
    ax4.set_title(
        f"Predicted: {pred_label}  |  Confidence: {confidence:.2%}  |  {decision}"
    )
    for bar, score in zip(bars, scores[::-1]):
        ax4.text(score + 1, bar.get_y() + bar.get_height() / 2, f"{score:.1f}%",
                  va="center", fontsize=9)

    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Report saved to:", out_path)

    if IN_COLAB:
        try:
            from google.colab import files
            files.download(out_path)
        except Exception:
            pass

    return out_path


##  Run the Full Pipeline

This ties every step above into a single function. Run the cell below and
follow the prompts.

In [ ]:
def run_pipeline(img_path=None):
    print("\n🩺  SKIN DISEASE DIAGNOSIS PIPELINE  🩺\n")

    # Step: user selects an image
    if img_path is None:
        img_path = get_image_path()

    # Step: preprocess
    img, img_array = preprocess_image(img_path)

    # Step: out-of-distribution sanity check
    is_skin_image, ood_reason, imagenet_top3 = check_is_skin_image(img)
    display_ood_result(is_skin_image, ood_reason, imagenet_top3)
    if not is_skin_image:
        print("\n🛑 Pipeline stopped — please try again with a valid skin image.")
        return {
            "image_path": img_path,
            "predicted_disease": None,
            "confidence": None,
            "decision": "Rejected — not a skin image",
            "top3": None,
            "ood_reason": ood_reason,
        }

    # Step: predict
    pred_label, confidence, preds = predict_disease(img_array)

    # Secondary sanity fallback: even if it passed the ImageNet gate, a very
    # low confidence from our own model is itself a signal something's off
    # (blurry photo, unusual angle, non-lesion skin, etc.)
    if confidence < SANITY_CONFIDENCE_THRESHOLD:
        print(f"⚠️  Note: model confidence ({confidence:.1%}) is very low even "
              f"after passing the sanity check — treat this result with caution.")

    # Step: display prediction + decision
    decision = display_prediction(pred_label, confidence)

    # Step: top-3
    top3 = get_top3(preds)
    display_top3(top3)

    # Step: Grad-CAM
    pred_idx = class_labels.index(pred_label)
    cam = generate_grad_cam(img_array, pred_idx)
    display_grad_cam(img, cam, pred_label, confidence)

    # Step: save report
    save_diagnostic_report(img, cam, top3, pred_label, confidence, decision)

    print("\n✅ Pipeline complete.")
    return {
        "image_path": img_path,
        "predicted_disease": pred_label,
        "confidence": confidence,
        "decision": decision,
        "top3": top3,
    }

# Run it:
results = run_pipeline()
